In [0]:
from pyspark.sql.functions import col, trim, lower, regexp_replace, when, current_timestamp

In [0]:
# Configurações do caminho
BRONZE_TABLE = "openbrew.brew_bronze.breweries"
SILVER_TABLE = "openbrew.brew_silver.breweries"

In [0]:
# Leitura dos dados da Bronze
df_bronze = spark.read.table(BRONZE_TABLE)

In [0]:
# Transformação e Padronização
df_silver = df_bronze.select(
    col("id").alias("brewery_id"),
    trim(col("name")).alias("brewery_name"),
    when(col("brewery_type").isNull(), "unknown")
    .otherwise(lower(col("brewery_type")))
    .alias("brewery_type"),
    trim(col("street")).alias("street_address"),
    trim(col("city")).alias("city"),
    trim(col("state_province")).alias("state"),
    col("postal_code"),
    col("country"),
    col("longitude").cast("double"),
    col("latitude").cast("double"),
    regexp_replace(col("phone"), "[^0-9]", "").alias("phone_number"),
    lower(col("website_url")).alias("website_url"),
    col("ingestion_timestamp").alias("original_ingestion_at"),
    current_timestamp().alias("processed_at")
)

In [0]:
# 4. Filtro de Qualidade (Remove registros que não possuem o ID ou o Nome)
df_silver = df_silver.filter("brewery_id IS NOT NULL AND brewery_name IS NOT NULL")

In [0]:
# Escrita na Camada Silver
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"Camada Silver atualizada com sucesso: {SILVER_TABLE}")